
# 영화 코퍼스 기반 워드 임베딩 편향성(WEAT) 분석

## 프로젝트 목표

주어진 영화 코퍼스를 바탕으로 다음 평가 항목을 충족한다.

1. 영화 코퍼스로 Word2Vec 워드 임베딩 모델을 구축한다.
2. `most_similar()` 결과를 확인하여 의미적으로 타당한지 검증한다.
3. 영화 장르별 target 단어와 예술영화/일반영화 attribute 단어 집합을 생성한다.
4. 중복 단어를 제거하여 각 개념축을 대표하는 단어 집합을 구성한다.
5. WEAT score를 계산하고 시각화한다.
6. 장르별 예술영화/일반영화 편향성을 해석한다.

> **Colab 제출용 노트북**
>
> 데이터 다운로드 → 전처리 → Word2Vec → most_similar → TF-IDF 단어셋 → WEAT → 시각화까지 한 번에 재현할 수 있도록 구성하였다.


## 1. Colab 환경 준비

In [ ]:

# Colab에서 최초 1회 실행
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk fonts-nanum
!pip -q install gdown konlpy gensim scikit-learn matplotlib seaborn numpy pandas


In [ ]:

import os
import re
import itertools
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

# Java / 한글 폰트 설정
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

print("환경 준비 완료")


## 2. 영화 코퍼스 다운로드 및 압축 해제

Google Drive의 코퍼스를 Colab의 `/content/w2v`에 저장한다. 루트(`/`)가 아닌 `/content`를 사용하여 읽기 전용 파일 시스템 오류를 피한다.

In [ ]:

import gdown

BASE_DIR = "/content/w2v"
ZIP_PATH = os.path.join(BASE_DIR, "synopsis.zip")
DATA_DIR = os.path.join(BASE_DIR, "synopsis")

os.makedirs(BASE_DIR, exist_ok=True)

FILE_ID = "19NmxfaX7tJn6dAGvq-EM1wfw63momOq0"

gdown.download(
    id=FILE_ID,
    output=ZIP_PATH,
    quiet=False
)

print("압축 파일:", ZIP_PATH)
print("존재 여부:", os.path.exists(ZIP_PATH))


In [ ]:

# 압축 해제
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)

os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print("압축 해제 완료")
print("상위 파일/폴더:")
for name in os.listdir(DATA_DIR)[:30]:
    print(" -", name)


## 3. 데이터 구조 확인

코퍼스 파일이 실제로 존재하는지 먼저 확인한다.

In [ ]:

for root, dirs, files in os.walk(DATA_DIR):
    print(f"\n[DIR] {root}")
    for f in files[:20]:
        print("  ", f)


## 4. 분석 설정

In [ ]:

GENRE_FILES = {
    "SF": "synopsis_SF.txt",
    "가족": "synopsis_가족.txt",
    "공포": "synopsis_공포(스릴러).txt",
    "다큐멘터리": "synopsis_다큐멘터리.txt",
    "드라마": "synopsis_드라마.txt",
    "멜로로맨스": "synopsis_멜로로맨스.txt",
    "미스터리": "synopsis_미스터리.txt",
    "범죄": "synopsis_범죄.txt",
    "애니메이션": "synopsis_애니메이션.txt",
    "액션": "synopsis_액션.txt",
    "코미디": "synopsis_코미디.txt",
    "판타지": "synopsis_판타지.txt",
}

ART_FILE = "synopsis_art.txt"
GENERAL_FILE = "synopsis_gen.txt"

N_TARGET_WORDS = 15
N_ATTRIBUTE_WORDS = 15
MIN_TOKEN_LEN = 2

print("데이터 경로:", DATA_DIR)
print("장르 수:", len(GENRE_FILES))


## 5. 데이터 로드 및 형태소 분석

In [ ]:

from konlpy.tag import Okt

tokenizer = Okt()
print("형태소 분석기: Okt")

def load_lines(filepath):
    path = os.path.join(DATA_DIR, filepath)
    if not os.path.exists(path):
        print(f"[경고] 파일 없음: {path}")
        return []
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize_nouns(text):
    nouns = tokenizer.nouns(text)
    return [w for w in nouns if len(w) >= MIN_TOKEN_LEN]

def build_corpus():
    print("코퍼스 로딩 및 형태소 분석 중...")

    genre_raw = {g: load_lines(f) for g, f in GENRE_FILES.items()}
    art_raw = load_lines(ART_FILE)
    gen_raw = load_lines(GENERAL_FILE)

    genre_tokens = {
        g: [tokenize_nouns(clean_text(line)) for line in lines]
        for g, lines in genre_raw.items()
    }
    art_tokens = [tokenize_nouns(clean_text(line)) for line in art_raw]
    gen_tokens = [tokenize_nouns(clean_text(line)) for line in gen_raw]

    return genre_tokens, art_tokens, gen_tokens

genre_tokens, art_tokens, gen_tokens = build_corpus()

print("장르별 문서 수")
for g, docs in genre_tokens.items():
    print(f"{g:8s}: {len(docs)}")

print("예술영화:", len(art_tokens))
print("일반영화:", len(gen_tokens))


## 6. Word2Vec 워드 임베딩 모델 구축

In [ ]:

all_sentences = []

for toks in genre_tokens.values():
    all_sentences.extend(toks)

all_sentences.extend(art_tokens)
all_sentences.extend(gen_tokens)

all_sentences = [s for s in all_sentences if len(s) > 0]

print("학습 문장 수:", len(all_sentences))

w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,
    workers=4,
    epochs=10,
    seed=42
)

print("Word2Vec 모델 학습 완료!")
print("Vocabulary size:", len(w2v_model.wv))


## 7. `most_similar()` 의미적 검증

아래 결과를 보고 영화 코퍼스의 문맥상 연관성이 높은 단어가 나오는지 확인한다.

In [ ]:

def sanity_check(model, query_words=("영화", "사랑", "액션"), topn=10):
    results = {}
    for w in query_words:
        if w in model.wv:
            sims = model.wv.most_similar(w, topn=topn)
            results[w] = sims
            print(f"\n[most_similar] '{w}'")
            for word, score in sims:
                print(f"  {word:12s} {score:.3f}")
        else:
            print(f"[안내] '{w}'는 vocabulary에 없습니다.")
    return results

similarity_results = sanity_check(w2v_model)


## 8. Target / Attribute 단어 집합 생성

TF-IDF 상위 단어를 후보로 사용하고, Word2Vec vocabulary에 존재하는 단어만 남긴다. 예술/일반 attribute 사이의 중복을 먼저 제거한 뒤, 장르 target에서도 attribute와 겹치는 단어를 제거한다.

In [ ]:

def get_tfidf_top_words(model, doc_list_of_token_lists, top_n=15, exclude=None):
    exclude = exclude or set()
    docs = [" ".join(toks) for toks in doc_list_of_token_lists if toks]

    if not docs:
        return []

    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(docs)

    scores = np.asarray(tfidf.sum(axis=0)).flatten()
    vocab = vectorizer.get_feature_names_out()

    ranked = sorted(
        zip(vocab, scores),
        key=lambda x: -x[1]
    )

    result = []

    for word, _ in ranked:
        if word in exclude:
            continue
        if word not in model.wv:
            continue

        result.append(word)

        if len(result) >= top_n:
            break

    return result


art_raw = get_tfidf_top_words(
    w2v_model,
    art_tokens,
    top_n=N_ATTRIBUTE_WORDS * 2
)

gen_raw = get_tfidf_top_words(
    w2v_model,
    gen_tokens,
    top_n=N_ATTRIBUTE_WORDS * 2
)

# 예술/일반 attribute 간 중복 제거
overlap = set(art_raw) & set(gen_raw)

attr_art = [
    w for w in art_raw
    if w not in overlap
][:N_ATTRIBUTE_WORDS]

attr_general = [
    w for w in gen_raw
    if w not in overlap
][:N_ATTRIBUTE_WORDS]

# target은 attribute 단어와 겹치지 않도록 제거
attr_all = set(attr_art) | set(attr_general)

genre_targets = {}

for genre, toks in genre_tokens.items():
    words = get_tfidf_top_words(
        w2v_model,
        toks,
        top_n=N_TARGET_WORDS * 2,
        exclude=attr_all
    )
    genre_targets[genre] = words[:N_TARGET_WORDS]

print("예술영화 attribute")
print(attr_art)

print("\n일반영화 attribute")
print(attr_general)

print("\n장르별 target")
for genre, words in genre_targets.items():
    print(f"[{genre}] {words}")


## 9. 단어 집합 중복 및 크기 검증

In [ ]:

print("예술 attribute 수:", len(attr_art))
print("일반 attribute 수:", len(attr_general))

print(
    "예술/일반 attribute 중복:",
    set(attr_art) & set(attr_general)
)

for genre, words in genre_targets.items():
    duplicates = set(words) & attr_all
    print(
        f"{genre:8s} | target={len(words):2d} | "
        f"attribute와 중복={len(duplicates)}"
    )


## 10. WEAT 계산

In [ ]:

def cos_sim(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return np.nan
    return np.dot(a, b) / denominator

def association(word, A, B, model):
    wv = model.wv[word]

    sims_a = [
        cos_sim(wv, model.wv[a])
        for a in A
        if a in model.wv
    ]

    sims_b = [
        cos_sim(wv, model.wv[b])
        for b in B
        if b in model.wv
    ]

    sims_a = [x for x in sims_a if not np.isnan(x)]
    sims_b = [x for x in sims_b if not np.isnan(x)]

    if not sims_a or not sims_b:
        return np.nan

    return np.mean(sims_a) - np.mean(sims_b)

def weat_effect_size(X, Y, A, B, model):
    X = [w for w in X if w in model.wv]
    Y = [w for w in Y if w in model.wv]

    if not X or not Y:
        return np.nan

    sx = np.array([
        association(w, A, B, model)
        for w in X
    ])

    sy = np.array([
        association(w, A, B, model)
        for w in Y
    ])

    sx = sx[~np.isnan(sx)]
    sy = sy[~np.isnan(sy)]

    if len(sx) == 0 or len(sy) == 0:
        return np.nan

    mean_diff = sx.mean() - sy.mean()
    pooled_std = np.concatenate([sx, sy]).std()

    if pooled_std == 0:
        return np.nan

    return mean_diff / pooled_std

def compute_weat_matrix(
    genre_targets,
    attr_art,
    attr_general,
    model
):
    genres = list(genre_targets.keys())
    matrix = pd.DataFrame(
        index=genres,
        columns=genres,
        dtype=float
    )

    for g1, g2 in itertools.product(genres, genres):
        if g1 == g2:
            matrix.loc[g1, g2] = 0.0
            continue

        matrix.loc[g1, g2] = weat_effect_size(
            genre_targets[g1],
            genre_targets[g2],
            attr_art,
            attr_general,
            model
        )

    return matrix

def compute_single_axis_bias(
    genre_targets,
    attr_art,
    attr_general,
    model
):
    scores = {}

    for genre, words in genre_targets.items():
        vals = [
            association(
                w,
                attr_art,
                attr_general,
                model
            )
            for w in words
            if w in model.wv
        ]

        vals = [
            v for v in vals
            if not np.isnan(v)
        ]

        scores[genre] = np.mean(vals) if vals else np.nan

    return scores

weat_matrix = compute_weat_matrix(
    genre_targets,
    attr_art,
    attr_general,
    w2v_model
)

single_scores = compute_single_axis_bias(
    genre_targets,
    attr_art,
    attr_general,
    w2v_model
)

print("WEAT matrix")
display(weat_matrix.round(3))

print("\n장르별 예술/일반 편향 점수")
display(
    pd.Series(single_scores, name="WEAT")
    .sort_values()
    .round(3)
)


## 11. WEAT 시각화 ① 장르 간 WEAT Heatmap

In [ ]:

heatmap_path = "/content/weat_heatmap.png"

plt.figure(figsize=(11, 8))

sns.heatmap(
    weat_matrix.astype(float),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title(
    "장르 간 WEAT Score\n"
    "(양수: 행 장르가 예술영화 쪽 / 음수: 일반영화 쪽)"
)
plt.xlabel("비교 대상 장르")
plt.ylabel("기준 장르")
plt.tight_layout()

plt.savefig(
    heatmap_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("저장:", heatmap_path)


## 12. WEAT 시각화 ② 장르별 예술/일반 편향

In [ ]:

bar_path = "/content/weat_genre_bias_bar.png"

scores = pd.Series(
    single_scores,
    name="WEAT"
).sort_values()

plt.figure(figsize=(11, 7))

bars = plt.barh(
    scores.index,
    scores.values
)

plt.axvline(
    0,
    color="black",
    linewidth=0.8
)

plt.xlabel(
    "예술영화(+)  <----->  일반영화(-)"
)

plt.ylabel("영화 장르")
plt.title(
    "장르별 예술/일반 영화 편향성 (WEAT 기반)"
)

# 막대 끝에 점수 표시
for bar, value in zip(bars, scores.values):
    offset = 0.01 if value >= 0 else -0.01
    ha = "left" if value >= 0 else "right"

    plt.text(
        value + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        va="center",
        ha=ha
    )

plt.tight_layout()

plt.savefig(
    bar_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("저장:", bar_path)


## 13. 제출용 결과 요약

In [ ]:

print("=" * 70)
print("WEAT 영화 코퍼스 분석 결과")
print("=" * 70)

print("\n[1] Word2Vec")
print("Vocabulary:", len(w2v_model.wv))
print("학습 문장 수:", len(all_sentences))

print("\n[2] Attribute")
print("예술영화:", attr_art)
print("일반영화:", attr_general)

print("\n[3] Target")
for genre, words in genre_targets.items():
    print(f"{genre}: {words}")

print("\n[4] 장르별 WEAT")
for genre, score in sorted(
    single_scores.items(),
    key=lambda x: x[1]
):
    print(f"{genre:10s}: {score:+.3f}")

print("\n[5] 생성된 시각화")
print(heatmap_path)
print(bar_path)



## 14. 루브릭 자체 점검

### 루브릭 1
- 영화 코퍼스 기반 Word2Vec 모델 구축
- `most_similar()` 결과 출력
- 결과의 의미적 타당성을 사람이 확인

### 루브릭 2
- 장르별 target 단어 집합 생성
- 예술영화 / 일반영화 attribute 집합 생성
- TF-IDF 기반 대표 단어 추출
- attribute 간 중복 제거
- target과 attribute 간 중복 제거

### 루브릭 3
- WEAT effect size 계산
- 전체 장르 쌍 WEAT matrix 생성
- Heatmap 시각화
- 장르별 단일 편향 점수 계산
- Bar chart 시각화

### 최종 해석
WEAT 점수의 양수/음수 방향과 크기를 확인하고,
`most_similar()` 및 생성된 target/attribute 단어 집합을 함께 검토하여
영화 장르별 결과가 영화 코퍼스의 의미적 특성과 상식적으로 부합하는지 판단한다.

> **주의:** WEAT의 방향과 값 자체만으로 "정답"을 단정하지 않는다.
> 실제 코퍼스의 단어 구성, Word2Vec 학습 결과, target/attribute 집합을 함께 근거로 해석한다.


In [ ]:

# 제출용 산출물 ZIP 생성
import shutil

output_dir = "/content/weat_submission"
os.makedirs(output_dir, exist_ok=True)

shutil.copy2(heatmap_path, output_dir)
shutil.copy2(bar_path, output_dir)

print("제출용 시각화 파일:")
print(os.listdir(output_dir))
